[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/03_tree_models/03_tree_models.ipynb)

# 03. 결정 트리와 랜덤 포레스트 — 학습, 평가, 튜닝

[02_preprocessing](../02_preprocessing/02_preprocessing.ipynb)에서 만든 `X`, `y`로 실제 모델을
학습시킵니다. 여기서 쓰는 **[결정 트리](../../../glossary.md#decision-tree)(Decision Tree)** 와 **[랜덤 포레스트](../../../glossary.md#random-forest)(Random Forest)** 는
표 형태 데이터에서 가장 널리 쓰이는 모델입니다.

## 왜 트리 모델인가

`ml-curriculum`에서 배운 선형 회귀·[로지스틱 회귀](../../../glossary.md#logistic-regression)·신경망과 비교하면 이런 장점이 있습니다.

| | 트리 모델 | 선형 모델 / 신경망 |
|---|---|---|
| [스케일링](../../../glossary.md#scaling) | **불필요** | 필수 |
| 범주형·수치형 혼합 | 자연스럽게 처리 | 인코딩 필요 |
| 비선형 관계 | **자동으로 학습** | 직접 항을 만들어줘야 함 |
| 변수 간 상호작용 | **자동으로 학습** | 직접 만들어줘야 함 |
| [이상치](../../../glossary.md#outlier) | **둔감** | 민감 |
| 해석 | **규칙을 그대로 볼 수 있음** | 어려움 |

특히 **표 데이터에서는 딥러닝보다 트리 계열이 더 좋은 성능을 내는 경우가 많습니다.**
이미지·텍스트에서는 딥러닝이 압도적이지만, 컬럼마다 의미가 다른 표 데이터에서는
그렇지 않습니다. 04번에서 신경망과 직접 비교해봅니다.

## 이 노트북의 구성

**1부에서 택시(회귀)로 모델링 전 과정을 한 바퀴 돌리고, 2부에서 타이타닉(분류)으로 넘어갑니다.**
트리 모델의 원리·[과적합](../../../glossary.md#overfitting)·교차검증·튜닝은 두 경우가 똑같습니다. **평가 지표만 완전히 다릅니다.**

| | 데이터 | 다루는 것 |
|---|---|---|
| **1부** | `trips` (회귀) | 트리의 분할 원리(분산) → MAE/RMSE/R² → 과적합 → 랜덤 포레스트 → **[변수중요도](../../../glossary.md#feature-importance)로 [데이터 누출](../../../glossary.md#data-leakage) 찾기** → 교차검증 → `GridSearchCV` → [잔차](../../../glossary.md#residual) 점검 |
| **2부** | `titanic` (분류) | [지니](../../../glossary.md#gini) 불순도 → **정확도의 함정**, [혼동 행렬](../../../glossary.md#confusion-matrix), 정밀도·재현율·F1 → 계층화 교차검증 → 튜닝 결과를 읽는 법 → 순열 중요도 |

1부의 하이라이트는 **`fare`가 변수중요도의 88%를 차지하는 것을 보고 데이터 누출을 잡아내는 장면**이고,
2부의 하이라이트는 **"전부 사망"이라고 찍어도 정확도 64.7%가 나오는 장면**입니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`). 아래쪽 셀은 위쪽 셀에서 만든
  변수를 그대로 쓰기 때문에, 중간부터 실행하면 `NameError`가 납니다.
- **실행 결과는 저장되어 있지 않습니다.** 코드 셀 아래가 비어 있는 것이 정상이고,
  직접 실행해야 표와 그래프가 나타납니다.
- 본문에 적힌 숫자(예: "MAE 8.33분")는 **실행하면 나오는 값**입니다. 글을 읽으면서
  그 숫자가 어느 셀의 출력인지 짚어보면 이해가 빠릅니다.
- 코드는 **그대로 실행만 해도 되지만**, 숫자를 바꿔 다시 실행해보는 것이 가장 좋은 연습입니다.
- **pandas 문법이 막히면** [00_pandas_for_tabular](../00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에
  이 시리즈에서 쓰는 문법만 모아뒀습니다(`pd.to_datetime`, `get_dummies`, `dropna` …). 사전처럼 찾아보세요.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib

### 준비 셀 — 라이브러리와 한글 폰트

아래 셀은 네 노트북에 공통으로 들어가는 준비 코드입니다. **내용을 이해할 필요는 없고 그냥
실행**하면 됩니다. `numpy`·`pandas`·`matplotlib`·`seaborn`을 불러오고, 그래프의 한글이
깨지지 않게 폰트를 잡고, 결과가 매번 같도록 무작위 시드(`RANDOM_STATE = 42`)를 고정합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

### 데이터 불러오기

01·02번과 같은 코드입니다. 예측 대상인 `duration`과 [파생 변수](../../../glossary.md#feature-engineering)를 만들어 둡니다.

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

print("trips:", trips.shape)

02번 1부에서 만든 `prepare_trips()`를 그대로 가져옵니다. (`prepare_titanic()`은 2부에서)

In [ ]:
def prepare_trips(raw):
    """[회귀] 택시 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    df = raw[(raw["duration"] > 0) & (raw["speed"] < 60)].copy()   # 이상치 제거
    df = df.drop(columns=["pickup", "dropoff",                     # 시각 자체는 weekday/hour로 대체
                          "pickup_zone", "dropoff_zone",           # 범주가 200개 이상이라 제외
                          "speed",                                 # duration으로 계산한 값 → 정답 누출
                          "total"])                                # fare+tip+tolls의 합 → 중복
    df = df.dropna()                                               # 결측치 행 제거
    df = pd.get_dummies(df, columns=["color", "payment",
                                     "pickup_borough", "dropoff_borough"],
                        drop_first=True)                           # 범주형 → 0/1
    X = df.drop(columns="duration")
    y = df["duration"]
    return X, y

학습용과 검증용으로 나눕니다. 02번 6절에서 본 `train_test_split`이고,
**회귀 문제라 `stratify`는 쓰지 않습니다**(클래스 비율이라는 개념이 없으므로).

In [ ]:
from sklearn.model_selection import train_test_split

X_reg, y_reg = prepare_trips(trips)       # 회귀: 이동 시간(분) 예측

X_train, X_valid, y_train, y_valid = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

print(f"[회귀] 학습 {X_train.shape}  검증 {X_valid.shape}")

> **스케일링을 하지 않았습니다.** 트리 모델은 각 컬럼을 따로 보며 "이 값보다 큰가/작은가"만
> 판단하기 때문에, 컬럼의 단위가 달라도 아무 영향이 없습니다. 04번의 신경망에서는 반드시 필요합니다.

---

# 1부. 택시로 회귀 모델 만들기

`trips` 하나만 봅니다. 02번 1부에서 만든 `X_reg`, `y_reg`로 **이동 시간(분)을 예측**합니다.

| 절 | 내용 |
|---|---|
| 1 | 트리가 데이터를 나누는 원리 — 분산이 가장 많이 줄어드는 질문 |
| 2~3 | 회귀 평가 지표(MAE·RMSE·R²)와 기준선, 과적합 |
| 4 | 랜덤 포레스트 |
| 5 | **변수중요도 — 여기서 데이터 누출을 잡아냅니다** |
| 6~7 | [교차 검증](../../../glossary.md#cross-validation), `GridSearchCV` |
| 8 | 잔차로 모델 점검하기 |

타이타닉은 2부에서 불러옵니다.

---

## 1. 결정 트리는 어떻게 예측하는가

결정 트리는 **질문을 연달아 던져 데이터를 좁혀가는 모델**입니다.
"요금이 15.55달러 이하인가?" → "8.75달러 이하인가?" → … 이런 식으로 내려가다가,
더 이상 나눌 수 없으면 그 자리에 남은 데이터로 답을 냅니다.

- **회귀**(1부): 그 자리에 남은 운행들의 **평균** — "이 구간의 운행은 평균 9.6분"
- **분류**(2부): 그 자리의 **다수결** — "10명 중 8명이 생존했으니 생존"

깊이 2짜리 작은 트리를 학습시켜 직접 그려봅시다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

small_tree = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE)
small_tree.fit(X_train, y_train)

plt.figure(figsize=(16, 7))
plot_tree(small_tree, feature_names=X_train.columns,
          filled=True, rounded=True, fontsize=10)
plt.show()

각 상자에 적힌 것을 읽는 법입니다.

| 항목 | 의미 |
|---|---|
| `fare <= 15.55` | **분할 조건.** 참이면 왼쪽, 거짓이면 오른쪽 |
| `squared_error = 133.74` | **불순도.** 이 노드에 남은 `duration` 값들이 흩어진 정도(분산) |
| `samples = 5068` | 이 노드에 도달한 데이터 수 |
| `value = 14.44` | **이 노드의 예측값** = 남은 운행들의 평균 이동 시간(분) |
| 색깔 | 진할수록 예측값이 큼 |

### 무엇을 기준으로 나누는가 — 회귀는 분산

트리는 **"나눈 뒤 양쪽의 값이 최대한 비슷해지는" 질문**을 고릅니다. "비슷한 정도"는 **분산**으로 잽니다.
분산이 작다는 것은 그 그룹의 값들이 평균 가까이 몰려 있다는 뜻이고, 그러면 평균으로 예측했을 때
잘 맞습니다.

루트에서 실제로 얼마나 줄어드는지 직접 계산해봅시다.

In [ ]:
# 루트 분할(fare <= 15.55)이 분산을 얼마나 줄이는가
mask = X_train["fare"] <= 15.55

left, right = y_train[mask], y_train[~mask]
n = len(y_train)

before = y_train.var(ddof=0)
after = (len(left) / n) * left.var(ddof=0) + (len(right) / n) * right.var(ddof=0)

print(f"나누기 전 분산   : {before:7.2f}  (n={n})")
print(f"왼쪽(요금 낮음)  : {left.var(ddof=0):7.2f}  (n={len(left)}, 평균 {left.mean():.2f}분)")
print(f"오른쪽(요금 높음): {right.var(ddof=0):7.2f}  (n={len(right)}, 평균 {right.mean():.2f}분)")
print(f"나눈 뒤 가중 평균: {after:7.2f}")
print(f"분산 감소량      : {before - after:7.2f}")

**트리는 모든 컬럼의 모든 분할 지점을 시도해보고, 이 감소량이 가장 큰 것을 고릅니다.**
그리고 나뉜 각 조각에서 같은 일을 반복합니다.

- **회귀**의 기준: 분산(MSE) — `criterion="squared_error"` (기본값). 위에서 계산한 것이 이것입니다
- **분류**의 기준: 지니 불순도 또는 엔트로피 — `criterion="gini"` / `"entropy"` (2부에서 다룹니다)

> **이 트리는 네 갈래 전부 `fare` 하나만 쓰고 있습니다.** 이동 시간을 정하는 것은 거리일 텐데
> 요금이 먼저 선택됐습니다. 어딘가 이상하죠. **왜 그런지는 5절에서 밝혀집니다.**

> 매번 "지금 이 순간 가장 좋은 분할"만 고르기 때문에, 전체적으로 최적인 트리를 찾는다는 보장은
> 없습니다(**탐욕적 알고리즘**). 이것이 트리 하나의 한계이고, 뒤에서 볼 랜덤 포레스트가
> 이 문제를 완화합니다.

---

## 2. 회귀 모델 학습과 평가

먼저 결정 트리로 이동 시간을 예측해봅시다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=RANDOM_STATE)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_valid)

print("실제값:", y_valid.values[:5].round(2))
print("예측값:", y_pred[:5].round(2))

### 회귀 평가 지표

예측이 얼마나 맞았는지를 숫자 하나로 요약하는 방법이 여러 가지 있고, **각각 다른 것을 말해줍니다.**

| 지표 | 계산 | 단위 | 특징 |
|---|---|---|---|
| **MAE** (평균 절대 오차) | 평균( \|실제 - 예측\| ) | 원래 단위(분) | 해석이 직관적. 이상치에 둔감 |
| **MSE** ([평균 제곱 오차](../../../glossary.md#mse)) | 평균( (실제-예측)² ) | 단위² | 큰 오차에 큰 벌점. 학습에 주로 사용 |
| **RMSE** | √MSE | 원래 단위(분) | MSE를 원래 단위로 되돌림 |
| **R²** (결정 계수) | 1 - (오차²합 / 분산합) | 없음 | **1에 가까울수록 좋음.** 0이면 "평균으로 찍는 것과 같음" |

**MAE와 RMSE의 차이가 핵심입니다.**

- 10분 틀린 예측 하나 vs 1분씩 틀린 예측 열 개 → **MAE는 같게** 봅니다
- **RMSE는 전자를 훨씬 나쁘게** 봅니다 (제곱하기 때문)

"크게 틀리는 것이 특히 곤란한" 문제라면 RMSE를, "평균적으로 얼마나 틀리는가"가 궁금하면 MAE를 봅니다.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)   # np.sqrt: 제곱근. MSE에 씌우면 원래 단위(분)로 돌아온다
r2 = r2_score(y_valid, y_pred)

print(f"MAE  : {mae:7.3f} 분")
print(f"MSE  : {mse:7.3f}")
print(f"RMSE : {rmse:7.3f} 분")
print(f"R²   : {r2:7.4f}")

### 이 숫자가 좋은 건가? — 기준선(baseline)을 먼저 만들자

"MAE 1.7분"이 좋은지 나쁜지는 그 자체로는 알 수 없습니다. **아무 것도 학습하지 않은 모델과
비교**해야 의미가 생깁니다. 가장 단순한 기준선은 **항상 평균으로 예측하는 것**입니다.

In [ ]:
baseline_mean = np.full(len(y_valid), y_train.mean())   # np.full(개수, 값): 같은 값으로 채운 배열. '항상 평균으로 찍는' 기준선을 만든다
baseline_median = np.full(len(y_valid), y_train.median())

print("기준선: 항상 평균으로 예측")
print(f"  MAE {mean_absolute_error(y_valid, baseline_mean):6.3f}  "
      f"RMSE {np.sqrt(mean_squared_error(y_valid, baseline_mean)):6.3f}  "
      f"R² {r2_score(y_valid, baseline_mean):7.4f}")
print()
print("기준선: 항상 중앙값으로 예측")
print(f"  MAE {mean_absolute_error(y_valid, baseline_median):6.3f}")
print()
print("결정 트리")
print(f"  MAE {mae:6.3f}  RMSE {rmse:6.3f}  R² {r2:7.4f}")

평균으로 찍으면 MAE 8.33분, 결정 트리는 1.69분입니다. **오차가 1/5로 줄었으니 모델이
실제로 뭔가를 배웠다**고 말할 수 있습니다.

기준선의 R²가 거의 0인 것도 확인해두세요. **R²는 "평균으로 찍는 것보다 얼마나 나은가"를
재는 지표**라서, 정의상 평균 예측의 R²는 0입니다. 모델이 그보다 못하면 **음수**가 나옵니다.

> MAE 기준선으로는 **중앙값**이 평균보다 낫습니다(7.70 < 8.33). MAE를 최소화하는 상수는
> 중앙값이고, MSE를 최소화하는 상수는 평균이기 때문입니다. 01번에서 본 "치우친 분포에서
> 평균과 중앙값이 다르다"는 성질이 여기서도 이어집니다.

---

## 3. 과적합 — 나무를 얼마나 깊게 키울 것인가

위에서 `max_depth`를 지정하지 않았습니다. 그러면 트리는 **더 나눌 수 없을 때까지** 자랍니다.
즉 잎 하나에 데이터 한 개만 남을 때까지 갑니다. 이것은 학습 데이터를 **통째로 외우는** 것과 같습니다.

깊이를 바꿔가며 학습 데이터와 검증 데이터의 오차를 함께 재봅시다.

In [ ]:
depths = [1, 2, 3, 5, 7, 10, 15, 20, None]
rows = []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({
        "max_depth": "제한 없음" if d is None else d,
        "실제 깊이": m.get_depth(),
        "잎 개수": m.get_n_leaves(),
        "학습 MAE": mean_absolute_error(y_train, m.predict(X_train)),
        "검증 MAE": mean_absolute_error(y_valid, m.predict(X_valid)),
    })

result = pd.DataFrame(rows)
result.round(3)

표만으로는 경향이 잘 안 보입니다. **두 곡선을 겹쳐 그리면** 어디서 갈라지는지 한눈에 보입니다.
(`max_depth=None`은 실제 깊이가 25였으므로 X축에 25로 찍습니다.)

In [ ]:
plot_depths = [1, 2, 3, 5, 7, 10, 15, 20, 25]   # None은 실제 깊이 25

plt.figure(figsize=(9, 5))
plt.plot(plot_depths, result["학습 MAE"], "o-", label="학습 데이터")
plt.plot(plot_depths, result["검증 MAE"], "s-", label="검증 데이터")
plt.axvline(10, color="gray", linestyle="--", linewidth=1)   # axvline: 지정한 x 위치에 세로 보조선을 긋는다
plt.text(10.3, 4, "최적 지점", color="gray")
plt.xlabel("트리 깊이 (max_depth)")
plt.ylabel("MAE (분)")
plt.title("트리가 깊어질수록 학습 오차는 계속 줄지만, 검증 오차는 다시 올라간다")
plt.legend()
plt.show()

**이 그래프가 과적합의 전형적인 모습입니다.**

| 깊이 | 학습 MAE | 검증 MAE | 상태 |
|---|---|---|---|
| 1 | 5.362 | 5.298 | **과소적합** — 질문 한 번으로는 부족 |
| 5 | 2.026 | 1.997 | 아직 여유 있음 |
| **10** | 0.802 | **1.491** | **가장 좋은 지점** |
| 15 | 0.106 | 1.617 | 과적합 시작 |
| 25(제한 없음) | **0.001** | 1.688 | **완전한 암기** |

깊이 제한을 없애면 **학습 오차가 0.001분(0.06초)** 까지 떨어집니다. 학습 데이터는 완벽하게
맞히는 것입니다. 그런데 **검증 오차는 오히려 나빠집니다.**

트리가 깊어지면서 "3월 12일 오후 3시에 브루클린에서 출발한 승객 2명짜리 운행은 14.3분"처럼
**일반화되지 않는 규칙**까지 만들어내기 때문입니다. 그런 손님은 다시 오지 않습니다.

**두 곡선이 벌어지기 시작하는 지점이 멈춰야 할 곳**입니다.

### 과적합을 막는 하이퍼파라미터

전부 **"나무를 덜 자라게 하는"** 방향으로 작동합니다.

| 파라미터 | 의미 | 기본값 | 조절 |
|---|---|---|---|
| `max_depth` | 트리의 최대 깊이 | `None`(무제한) | **작을수록 단순** |
| `min_samples_split` | 노드를 나누는 데 필요한 최소 샘플 수 | 2 | **클수록 단순** |
| `min_samples_leaf` | 잎에 남아야 할 최소 샘플 수 | 1 | **클수록 단순** |
| `max_leaf_nodes` | 잎의 최대 개수 | `None` | 작을수록 단순 |
| `max_features` | 각 분할에서 고려할 컬럼 수 | 전부 | 작을수록 단순 |
| `min_impurity_decrease` | 이만큼 불순도가 줄지 않으면 분할 안 함 | 0.0 | 클수록 단순 |

**`min_samples_leaf`가 특히 유용합니다.** 이 값을 5로 두면 "데이터 한두 개만 보고 만든 규칙"이
아예 생기지 않습니다. `max_depth`처럼 트리 전체를 일괄로 자르는 대신, 데이터가 많은 영역은
깊게 데이터가 적은 영역은 얕게 자라도록 **적응적으로** 조절해줍니다.

In [ ]:
tuned = DecisionTreeRegressor(
    max_depth=10, min_samples_split=10, min_samples_leaf=5, random_state=RANDOM_STATE
).fit(X_train, y_train)

for name, m in [("기본값 (무제한)", dt), ("조절 후", tuned)]:
    p = m.predict(X_valid)
    print(f"{name:16s} 깊이 {m.get_depth():2d}  잎 {m.get_n_leaves():5d}개  "
          f"검증 MAE {mean_absolute_error(y_valid, p):.3f}  R² {r2_score(y_valid, p):.4f}")

---

## 4. 랜덤 포레스트 — 나무 여러 개를 모으기

결정 트리 하나는 불안정합니다. 데이터가 조금만 달라져도 첫 분할이 바뀌고, 그러면
트리 전체가 완전히 달라집니다. **분산(variance)이 크다**고 말합니다.

**랜덤 포레스트는 트리를 수백 개 만들어 평균을 냅니다.** 각 트리는 서로 다르게 만듭니다.

1. **부트스트랩 샘플링(bagging)**: 각 트리는 원본에서 **복원 추출한** 데이터로 학습합니다.
   같은 크기지만 중복이 있고 빠진 데이터도 있어서, 트리마다 다른 데이터를 봅니다
2. **컬럼 무작위 선택**: 각 분할마다 **전체 컬럼 중 일부만** 후보로 씁니다.
   덕분에 강한 변수 하나가 모든 트리를 지배하는 것을 막습니다

**"제각각 틀리는 모델 여럿의 평균은 개별 모델보다 낫다"** 는 원리입니다.
트리 하나가 어떤 손님을 잘못 예측해도, 다른 트리들이 상쇄해줍니다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_valid)

comparison = pd.DataFrame({
    "MAE": [
        mean_absolute_error(y_valid, baseline_mean),
        mean_absolute_error(y_valid, dt.predict(X_valid)),
        mean_absolute_error(y_valid, tuned.predict(X_valid)),
        mean_absolute_error(y_valid, rf_pred),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_valid, baseline_mean)),
        np.sqrt(mean_squared_error(y_valid, dt.predict(X_valid))),
        np.sqrt(mean_squared_error(y_valid, tuned.predict(X_valid))),
        np.sqrt(mean_squared_error(y_valid, rf_pred)),
    ],
    "R²": [
        r2_score(y_valid, baseline_mean),
        r2_score(y_valid, dt.predict(X_valid)),
        r2_score(y_valid, tuned.predict(X_valid)),
        r2_score(y_valid, rf_pred),
    ],
}, index=["기준선(평균)", "결정 트리(기본)", "결정 트리(조절)", "랜덤 포레스트"]).round(4)
comparison

**랜덤 포레스트가 조절한 결정 트리보다도 낫습니다.** 게다가 [하이퍼파라미터](../../../glossary.md#hyperparameter)를 하나도 건드리지
않은 기본값입니다. 트리 하나를 정성껏 튜닝하는 것보다, 대충 만든 트리를 여러 개 모으는 쪽이
쉽고 강력합니다.

### 주요 하이퍼파라미터

결정 트리의 파라미터를 전부 그대로 쓸 수 있고, 여기에 두 개가 추가됩니다.

| 파라미터 | 의미 | 기본값 |
|---|---|---|
| `n_estimators` | 나무 개수 | 100 |
| `max_features` | 각 분할에서 고려할 컬럼 수 | 회귀 `1.0`(전부), 분류 `"sqrt"` |
| `n_jobs` | 병렬 처리 코어 수 (`-1`이면 전부) | `None` |
| `oob_score` | 부트스트랩에서 빠진 데이터로 자체 평가 | `False` |

**`n_estimators`는 클수록 좋지만 수익이 체감합니다.** 그리고 **많다고 과적합되지는 않습니다** —
평균을 내는 것이라 안정될 뿐입니다. 늘어나는 것은 학습 시간뿐입니다.

In [ ]:
for n in [1, 5, 10, 50, 100, 300]:
    m = RandomForestRegressor(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    print(f"나무 {n:3d}개 -> 검증 MAE {mean_absolute_error(y_valid, m.predict(X_valid)):.4f}")

나무 1개에서 10개로 갈 때 크게 좋아지고, 100개 이후로는 거의 변화가 없습니다.
**보통 100~300 사이면 충분**하고, 그 이상은 시간만 더 씁니다.

---

## 5. 변수중요도 — 모델이 무엇을 보고 있는가

트리 모델은 **어떤 컬럼이 예측에 얼마나 기여했는지** 알려줍니다.
`feature_importances_`는 그 컬럼이 만들어낸 **불순도 감소량의 총합**을 정규화한 값입니다
(전부 더하면 1).

In [ ]:
def plot_importance(model, feature_names, top=10, title="변수중요도"):
    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(8, top * 0.4 + 1))
    # barplot: 값 크기를 막대 길이로. y에 문자열을 주면 가로 막대가 된다
    sns.barplot(data=fi, x="importance", y="feature", palette="viridis")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return fi


fi_reg = plot_importance(rf, X_train.columns, top=10, title="이동 시간 예측 — 변수중요도")
fi_reg.round(4)

### 잠깐 — `fare`가 0.878입니다

`fare` 하나가 **전체 중요도의 88%** 를 차지합니다. `distance`(0.068)보다 13배 큽니다.
**이상합니다.** 이동 시간을 정하는 것은 거리와 교통 상황이지 요금이 아닙니다.

01번에서 배운 질문을 던져봅시다. **"예측하는 시점에 이 값을 알 수 있는가?"**

**뉴욕 택시 요금은 거리 요금 + 시간 요금으로 계산됩니다.** 막혀서 서 있는 동안에도
미터기가 올라갑니다. 즉 `fare` 안에는 **이동 시간 정보가 이미 들어 있습니다.**
그리고 요금은 **운행이 끝나야 확정됩니다.**

우리가 만들려는 것은 "출발할 때 도착 시간을 알려주는 모델"입니다. 그 시점에 알 수 있는 것은
거리·시각·요일·지역뿐입니다. `fare`, `tip`, `tolls`는 **전부 운행이 끝난 뒤에 확정되는 값**입니다.

02번에서 `speed`를 지운 것과 **똑같은 종류의 데이터 누출**인데, `speed`처럼 명백하지 않아서
여기까지 살아남았습니다. **변수중요도는 바로 이런 것을 잡아내라고 보는 것입니다.**

In [ ]:
# 사후 정보를 모두 제거하고 다시 학습
leaky_cols = ["fare", "tip", "tolls"]

X_train_fixed = X_train.drop(columns=leaky_cols)
X_valid_fixed = X_valid.drop(columns=leaky_cols)

rf_fixed = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf_fixed.fit(X_train_fixed, y_train)

pred_fixed = rf_fixed.predict(X_valid_fixed)

print("=== 누출 컬럼 포함 (겉보기 성능) ===")
print(f"MAE {mean_absolute_error(y_valid, rf_pred):.3f} 분   R² {r2_score(y_valid, rf_pred):.4f}")
print()
print("=== 누출 컬럼 제거 (실제로 쓸 수 있는 성능) ===")
print(f"MAE {mean_absolute_error(y_valid, pred_fixed):.3f} 분   R² {r2_score(y_valid, pred_fixed):.4f}")
print()
print(f"기준선(평균)        MAE {mean_absolute_error(y_valid, baseline_mean):.3f} 분")

**MAE가 1.19분에서 3.55분으로 3배 나빠졌습니다.** R²도 0.947 → 0.783.

실망스러워 보이지만 **이쪽이 진짜 성능입니다.** 앞의 1.19분은 "요금을 알려주면 시간을
맞힐 수 있다"는, 실제로는 쓸 수 없는 능력이었습니다.

그리고 3.55분도 나쁘지 않습니다. 기준선(8.33분)의 **절반 이하**니까요.
출발할 때 알 수 있는 정보만으로 도착 시간을 ±3.5분 안에 맞히는 모델입니다.

> **이것이 데이터 누출이 무서운 이유입니다.** 에러도 경고도 없고, 오히려 **성능이 좋아 보입니다.**
> 그대로 배포하면 실제 환경에서 성능이 반토막 나고, 원인을 찾기 어렵습니다.
> 변수중요도가 상식과 어긋날 때 반드시 멈춰서 확인해야 합니다.

In [ ]:
fi_fixed = plot_importance(rf_fixed, X_train_fixed.columns, top=10,
                           title="이동 시간 예측 — 누출 제거 후")
fi_fixed.round(4)

이제 **`distance`(0.805)가 압도적이고 `hour`(0.089), `weekday`(0.040)** 가 뒤를 잇습니다.
"거리가 시간을 결정하고, 시간대와 요일이 교통 상황을 통해 보정한다" — 상식과 일치합니다.

---

## 6. 교차 검증 — 한 번의 분할을 믿지 말 것

지금까지 **한 번 나눈 검증 데이터**로만 평가했습니다. 그런데 우연히 쉬운 데이터가 검증 쪽에
몰렸다면 점수가 과대평가됩니다. 검증 데이터가 작을수록 이 위험이 커집니다.

**k-겹 교차 검증(k-fold cross validation)** 은 데이터를 k조각으로 나눠, 각 조각을 한 번씩
검증용으로 쓰며 **k번 학습하고 평가**합니다.

```
fold 1: [검증][학습][학습][학습][학습]
fold 2: [학습][검증][학습][학습][학습]
fold 3: [학습][학습][검증][학습][학습]   → 5개 점수의 평균과 표준편차
fold 4: [학습][학습][학습][검증][학습]
fold 5: [학습][학습][학습][학습][검증]
```

**회귀에서는 `KFold`가 기본**입니다. 데이터에 순서가 있다면 `shuffle=True`가 필요합니다.
(분류에서는 클래스 비율을 맞추는 `StratifiedKFold`가 자동으로 쓰이는데, 2부에서 다룹니다.)

In [ ]:
from sklearn.model_selection import cross_val_score

# 회귀에서는 "오차가 작을수록 좋다"를 "점수가 클수록 좋다"로 맞추기 위해 음수를 씁니다
reg_scores = cross_val_score(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
)

print("MAE (부호 뒤집음):", (-reg_scores).round(4))
print(f"평균 {(-reg_scores).mean():.4f}  표준편차 {reg_scores.std():.4f}")

fold마다 MAE가 1.28 ~ 1.45분으로 흔들립니다. **한 번만 나눴다면 어느 값이 나올지 운에
달렸던 셈**입니다. 평균을 쓰면 훨씬 믿을 만하고, **표준편차는 그 추정이 얼마나 불안정한지**를
알려줍니다.

> `neg_`가 붙은 이유는 scikit-learn이 **"점수는 클수록 좋다"** 로 통일해 두었기 때문입니다.
> MAE는 작을수록 좋으므로 부호를 뒤집어 `neg_mean_absolute_error`로 제공합니다.
> 자주 쓰는 것들: `"accuracy"`, `"f1"`, `"roc_auc"`, `"r2"`,
> `"neg_mean_absolute_error"`, `"neg_root_mean_squared_error"`

---

## 7. `GridSearchCV` — 하이퍼파라미터를 체계적으로 찾기

지금까지는 `max_depth=10`, `min_samples_leaf=5` 같은 값을 손으로 골랐습니다.
**`GridSearchCV`는 후보 값들의 모든 조합을 교차 검증으로 시험해 가장 좋은 것을 찾아줍니다.**

누출 컬럼을 제거한 데이터(`X_train_fixed`)로 랜덤 포레스트를 튜닝해봅시다.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 2, 5],
}

gs_reg = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=3,                                  # 3겹 교차 검증
    scoring="neg_mean_absolute_error",
    n_jobs=-1,                             # 모든 코어 사용
)
gs_reg.fit(X_train_fixed, y_train)

final_reg_pred = gs_reg.predict(X_valid_fixed)

print(f"시험한 조합: {len(gs_reg.cv_results_['params'])}개 × 3겹 = "
      f"{len(gs_reg.cv_results_['params']) * 3}회 학습")
print("최적 조합:", gs_reg.best_params_)
print(f"교차 검증 MAE : {-gs_reg.best_score_:.3f} 분")
print(f"검증 데이터 MAE: {mean_absolute_error(y_valid, final_reg_pred):.3f} 분  "
      f"(기준선 {mean_absolute_error(y_valid, baseline_mean):.3f} 분)")
print(f"R²            : {r2_score(y_valid, final_reg_pred):.4f}")

| 속성 | 내용 |
|---|---|
| `best_params_` | 가장 좋았던 파라미터 조합 |
| `best_score_` | 그때의 **교차 검증** 평균 점수 |
| `best_estimator_` | 그 조합으로 **전체 학습 데이터에 다시 학습된** 모델 |
| `cv_results_` | 모든 조합의 상세 결과 (DataFrame으로 변환 가능) |

`cv_results_`를 열어보면 어떤 조합이 좋았는지 전부 볼 수 있습니다.

In [ ]:
results = pd.DataFrame(gs_reg.cv_results_)
cols = ["param_max_depth", "param_min_samples_leaf", "param_n_estimators",
        "mean_test_score", "std_test_score", "rank_test_score"]

# neg_mean_absolute_error라서 점수가 음수입니다 — 부호를 뒤집어 MAE로 읽습니다
out = results[cols].sort_values("rank_test_score").head(8).copy()
out["mean_test_score"] = -out["mean_test_score"]
out.round(4)

> #### 격자가 너무 커질 때
>
> 위 격자는 2 × 3 × 3 = 18조합 × 3겹 = **54번 학습**입니다. 파라미터를 하나 더 추가하면
> 곱셈으로 늘어납니다(**조합 폭발**).
>
> - **`RandomizedSearchCV`**: 모든 조합 대신 **무작위로 n개만** 시험합니다. 넓은 범위를
>   훑을 때 격자 탐색보다 효율적인 경우가 많습니다 (연습문제 1번)
> - **`HalvingGridSearchCV`**: 처음엔 적은 데이터로 많은 조합을 걸러내고, 살아남은 조합에만
>   데이터를 늘려가며 시험합니다
> - 넓은 범위를 성기게 훑어 유망한 구간을 찾고, 그 근처를 촘촘히 다시 훑는 **2단계 탐색**도 흔합니다

---

## 8. 잔차로 모델 점검하기

지표 하나로 끝내지 말고 **어디서 틀리는지**를 봐야 합니다.

In [ ]:
# 04번에서 신경망과 비교할 수 있도록 예측 결과를 남겨둡니다
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_valid, final_reg_pred, alpha=0.3, s=10)
lims = [0, max(y_valid.max(), final_reg_pred.max())]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("실제 이동 시간 (분)")
axes[0].set_ylabel("예측 이동 시간 (분)")
axes[0].set_title("실제 vs 예측 (완벽하면 빨간 선 위)")

residual = y_valid - final_reg_pred
sns.histplot(residual, bins=50, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("잔차 (실제 - 예측)")
axes[1].set_title(f"잔차 분포 (평균 {residual.mean():.3f})")

plt.tight_layout()
plt.show()

**잔차 그래프는 회귀 모델을 점검하는 표준 도구입니다.**

- 잔차가 **0을 중심으로 대칭**이면 편향 없이 예측하고 있다는 뜻입니다
- 왼쪽 그래프에서 긴 운행일수록 점이 빨간 선 아래로 처집니다 —
  **모델이 긴 운행을 실제보다 짧게 예측하는 경향**이 있습니다.
  긴 운행 데이터가 상대적으로 적어서 생기는 현상입니다


**1부 끝.** 출발 시점에 알 수 있는 정보만으로 도착 시간을 ±3.4분 안에 맞히는 모델입니다.

되짚어보면 이 노트북에서 가장 중요한 장면은 성능 숫자가 아니라 **변수중요도를 보고 `fare`를
의심한 순간**이었습니다. 그대로 배포했다면 검증 점수 MAE 1.19분을 믿고 나갔다가,
실제 환경에서 3배 나쁜 성능을 만나고 원인을 찾지 못했을 것입니다.

이제 타이타닉으로 넘어갑니다. **모델을 만드는 절차는 똑같고, 평가하는 방법이 완전히 다릅니다.**

---

# 2부. 타이타닉으로 분류 모델 만들기

트리를 학습시키고 과적합을 막고 튜닝하는 절차는 1부와 **완전히 같습니다.**
`Regressor`가 `Classifier`로 바뀔 뿐입니다. 달라지는 것은 두 가지입니다.

| | 1부 (회귀) | 2부 (분류) |
|---|---|---|
| 분할 기준 | 분산(MSE) | **지니 불순도** |
| 평가 지표 | MAE · RMSE · R² | **정확도 · 정밀도 · 재현율 · F1** |
| 교차 검증 | `KFold` | **`StratifiedKFold`** (클래스 비율 유지) |

**"정확도 하나만 보면 안 된다"** 는 것이 2부의 핵심입니다.

## 9. 타이타닉 준비

In [ ]:
def prepare_titanic(raw):
    """[분류] 타이타닉 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    q1, q3 = raw["fare"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    df = raw[(raw["fare"] >= lower) & (raw["fare"] <= upper)].copy()  # 이상치 제거
    df = df.drop(columns=["alive",                                   # survived와 같은 정보 → 정답 누출
                          "class", "embark_town",                    # pclass/embarked와 중복
                          "deck",                                    # 결측치가 77%
                          "adult_male"])                             # who와 중복
    df = df.dropna()
    df = pd.get_dummies(df, columns=["sex", "embarked", "who"], drop_first=True)
    X = df.drop(columns="survived")
    y = df["survived"]
    return X, y


titanic = sns.load_dataset("titanic")
X_clf, y_clf = prepare_titanic(titanic)

Xc_train, Xc_valid, yc_train, yc_valid = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_STATE, stratify=y_clf
)

print(f"[분류] 학습 {Xc_train.shape}  검증 {Xc_valid.shape}")
print(f"생존율: 학습 {yc_train.mean() * 100:.1f}%  검증 {yc_valid.mean() * 100:.1f}%")

---

## 10. 분류 트리와 지니 불순도

1부의 회귀 트리는 **분산**이 가장 많이 줄어드는 질문을 골랐습니다. 분류 트리도 똑같은데,
"섞여 있는 정도"를 재는 자가 다를 뿐입니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

small_tree = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
small_tree.fit(Xc_train, yc_train)

plt.figure(figsize=(16, 7))
plot_tree(small_tree,
          feature_names=Xc_train.columns,
          class_names=["사망", "생존"],
          filled=True, rounded=True, fontsize=10)
plt.show()

각 상자에 적힌 것을 읽는 법입니다.

| 항목 | 의미 |
|---|---|
| `who_man <= 0.5` | **분할 조건.** 참이면 왼쪽, 거짓이면 오른쪽 (0/1 컬럼이라 "성인 남성이 아닌가?") |
| `gini = 0.457` | **불순도.** 이 노드에 섞여 있는 정도 |
| `samples = 429` | 이 노드에 도달한 데이터 수 |
| `value = [277, 152]` | 클래스별 개수 [사망, 생존] |
| `class = 사망` | 다수결로 정한 이 노드의 예측 |
| 색깔 | 진할수록 한쪽 클래스로 치우침 |

### 불순도(impurity) — 무엇을 기준으로 나누는가

트리는 **"나눈 뒤 양쪽이 최대한 순수해지는" 질문**을 고릅니다. 순수하다는 것은
한쪽 클래스만 남았다는 뜻입니다.

**지니 불순도(Gini impurity)** 는 이렇게 계산합니다.

```
Gini = 1 - Σ (각 클래스의 비율)²
```

- 생존 50% / 사망 50% → `1 - (0.5² + 0.5²) = 0.5` (**최대로 섞임**)
- 생존 90% / 사망 10% → `1 - (0.9² + 0.1²) = 0.18`
- 생존 100% → `1 - 1² = 0` (**완전히 순수**)

직접 계산해보겠습니다.

In [ ]:
def gini(counts):
    """클래스별 개수 -> 지니 불순도"""
    counts = np.array(counts, dtype=float)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)


print("생존 50 / 사망 50 :", round(gini([50, 50]), 4))
print("생존 90 / 사망 10 :", round(gini([10, 90]), 4))
print("생존 100/ 사망  0 :", round(gini([0, 100]), 4))
print()
# np.bincount: 0,1,2… 각 정수가 몇 번 나오는지 센다 -> 클래스별 개수
print("학습 데이터 전체  :", round(gini(np.bincount(yc_train)), 4))

이 함수로 **실제 분할이 불순도를 얼마나 줄이는지** 계산해봅시다. 나눈 뒤의 불순도는
양쪽 조각을 **크기로 가중 평균** 내서 구합니다. 작은 조각이 순수해도 전체에 미치는 영향은 작기 때문입니다.

In [ ]:
# who_man 으로 나누면 불순도가 얼마나 줄어드는가
mask = Xc_train["who_man"] == 1

left = np.bincount(yc_train[~mask])   # 성인 남성이 아닌 쪽
right = np.bincount(yc_train[mask])   # 성인 남성인 쪽

n = len(yc_train)
n_left, n_right = (~mask).sum(), mask.sum()

before = gini(np.bincount(yc_train))
after = (n_left / n) * gini(left) + (n_right / n) * gini(right)

print(f"나누기 전 불순도     : {before:.4f}  (n={n})")
print(f"왼쪽(성인 남성 아님) : {gini(left):.4f}  (n={n_left})")
print(f"오른쪽(성인 남성)    : {gini(right):.4f}  (n={n_right})")
print(f"나눈 뒤 가중 평균    : {after:.4f}")
print(f"불순도 감소량        : {before - after:.4f}")

---

## 11. 분류 모델과 평가 지표

이번엔 타이타닉 생존 예측입니다. 회귀와 코드 구조는 같고, **모델 이름 끝이 `Classifier`** 이며
**평가 지표가 다릅니다.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

dt_clf = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(Xc_train, yc_train)
dt_clf_tuned = DecisionTreeClassifier(max_depth=5, min_samples_split=10,
                                      random_state=RANDOM_STATE).fit(Xc_train, yc_train)
rf_clf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_train, yc_train)

# 기준선: 무조건 다수 클래스(사망)로 예측
baseline_clf = np.zeros(len(yc_valid), dtype=int)

for name, pred in [
    ("기준선(전부 사망)", baseline_clf),
    ("결정 트리(기본)", dt_clf.predict(Xc_valid)),
    ("결정 트리(depth=5)", dt_clf_tuned.predict(Xc_valid)),
    ("랜덤 포레스트", rf_clf.predict(Xc_valid)),
]:
    print(f"{name:20s} 정확도 {accuracy_score(yc_valid, pred):.4f}")

### 정확도의 함정

**아무것도 학습하지 않고 "전부 사망"이라고만 찍어도 64.7%가 나옵니다.** 검증 데이터의
64.7%가 실제로 사망자이기 때문입니다.

이것이 **정확도(accuracy)만 봐서는 안 되는 이유**입니다. 클래스가 불균형할수록 심해집니다.
사기 거래가 0.1%인 데이터에서 "전부 정상"이라고 찍으면 정확도 **99.9%** 인 무용지물 모델이 됩니다.

### 혼동 행렬 (Confusion Matrix)

무엇을 어떻게 틀렸는지 보려면 **실제 vs 예측**을 표로 봐야 합니다.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred_clf = rf_clf.predict(Xc_valid)
cm = confusion_matrix(yc_valid, y_pred_clf)

print(cm)
print()

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["사망", "생존"]).plot(ax=ax, cmap="Blues")
plt.title("혼동 행렬 (랜덤 포레스트)")
plt.show()

```
                  예측: 사망    예측: 생존
  실제: 사망         92            27      ← 27명을 생존이라고 잘못 예측
  실제: 생존         18            47      ← 18명을 사망이라고 잘못 예측
```

각 칸에는 이름이 있습니다 (**"생존"을 양성(positive)으로 볼 때**).

| | 이름 | 개수 |
|---|---|---|
| 사망을 사망으로 | **TN** (True Negative) | 92 |
| 사망을 생존으로 | **FP** (False Positive, 거짓 경보) | 27 |
| 생존을 사망으로 | **FN** (False Negative, 놓침) | 18 |
| 생존을 생존으로 | **TP** (True Positive) | 47 |

### 정밀도 · 재현율 · F1

```
정확도 (Accuracy)  = (TP + TN) / 전체          전체 중 맞힌 비율
정밀도 (Precision) = TP / (TP + FP)            "생존"이라 한 것 중 진짜 생존 비율
재현율 (Recall)    = TP / (TP + FN)            실제 생존자 중 찾아낸 비율
F1                 = 정밀도와 재현율의 조화평균   둘의 균형
```

**정밀도와 재현율은 서로 상충합니다.**

- 재현율을 높이려면 → 조금만 의심스러워도 "생존"이라고 함 → 거짓 경보(FP) 증가 → 정밀도 하락
- 정밀도를 높이려면 → 확실할 때만 "생존"이라고 함 → 놓치는 경우(FN) 증가 → 재현율 하락

**어느 쪽이 중요한지는 문제에 달렸습니다.**

| 상황 | 중요한 것 | 이유 |
|---|---|---|
| 암 진단 | **재현율** | 놓치면(FN) 사람이 죽습니다. 오진(FP)은 재검사로 해결 |
| 스팸 필터 | **정밀도** | 중요한 메일이 스팸함에 가면(FP) 큰일. 스팸 몇 개 놓치는(FN) 건 참을 만함 |
| 사기 탐지 | 상황에 따라 | 놓치면 손실, 과하면 정상 고객 불편 |

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

tn, fp, fn, tp = cm.ravel()

print("직접 계산")
print(f"  정확도 = ({tp} + {tn}) / {cm.sum()}     = {(tp + tn) / cm.sum():.4f}")
print(f"  정밀도 = {tp} / ({tp} + {fp})       = {tp / (tp + fp):.4f}")
print(f"  재현율 = {tp} / ({tp} + {fn})       = {tp / (tp + fn):.4f}")
print()
print("scikit-learn")
print(f"  정밀도 {precision_score(yc_valid, y_pred_clf):.4f}  "
      f"재현율 {recall_score(yc_valid, y_pred_clf):.4f}  "
      f"F1 {f1_score(yc_valid, y_pred_clf):.4f}")

위 세 지표를 클래스별로 한 번에 보여주는 것이 `classification_report`입니다.

In [ ]:
print(classification_report(yc_valid, y_pred_clf, target_names=["사망", "생존"]))

`classification_report`는 **두 클래스 각각에 대해** 지표를 보여줍니다.
"사망"에 대한 정밀도 0.84는 "사망이라고 예측한 110명 중 92명이 실제 사망"이라는 뜻입니다.

맨 아래 두 줄도 알아두면 좋습니다.

- **macro avg**: 클래스별 지표의 **단순 평균**. 소수 클래스도 똑같은 비중으로 봅니다
- **weighted avg**: 클래스 크기로 **가중 평균**. 다수 클래스가 더 반영됩니다

**불균형 데이터에서는 macro avg를 봐야** 소수 클래스의 성능이 드러납니다.

> 예측 확률을 직접 다루고 싶다면 `predict_proba()`를 씁니다. 기본 임계값 0.5 대신 다른 값을
> 쓰면 정밀도-재현율 균형을 조절할 수 있습니다. 임계값 전체에 걸친 성능은
> **ROC 곡선과 AUC**(`roc_auc_score`)로 요약합니다.

In [ ]:
proba = rf_clf.predict_proba(Xc_valid)[:, 1]   # 생존 확률

for threshold in [0.3, 0.5, 0.7]:
    pred = (proba >= threshold).astype(int)
    print(f"임계값 {threshold}: "
          f"정밀도 {precision_score(yc_valid, pred):.3f}  "
          f"재현율 {recall_score(yc_valid, pred):.3f}  "
          f"F1 {f1_score(yc_valid, pred):.3f}")

임계값을 낮추면 재현율이 오르고 정밀도가 떨어집니다. 예상한 대로입니다.

---

## 12. 교차 검증 — 분류에서 달라지는 것

1부에서 회귀 모델에 `KFold`를 썼습니다. 분류에서는 **`cv=5`만 줘도 scikit-learn이 자동으로
`StratifiedKFold`를 씁니다** — 각 fold의 클래스 비율을 유지해줍니다(02번의 `stratify`와 같은 개념).

타이타닉은 학습 데이터가 429건뿐이라 fold마다 점수가 크게 흔들립니다.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    Xc_train, yc_train, cv=5, scoring="accuracy"
)

print("각 fold 점수:", scores.round(4))
print(f"평균 {scores.mean():.4f}  표준편차 {scores.std():.4f}")
print(f"범위 {scores.min():.4f} ~ {scores.max():.4f}")

**fold마다 0.756 ~ 0.826으로 7%p나 차이 납니다.** 한 번만 나눴다면 어느 값이 나올지
운에 달렸던 셈입니다. 평균을 쓰면 훨씬 믿을 만하고, **표준편차는 그 추정이 얼마나
불안정한지**를 알려줍니다.

1부의 회귀에서 본 것과 같은 이야기입니다. 다만 학습 데이터가 429건뿐이라 흔들림이 훨씬 큽니다.

### 교차 검증과 전처리를 함께 쓸 때 — `Pipeline`

02번 연습문제 2번에서 본 문제가 여기서 더 심각해집니다. 교차 검증은 데이터를 **매 fold마다
다르게 나누므로**, 스케일링을 미리 해두면 fold마다 누출이 반복됩니다.

`Pipeline`으로 전처리와 모델을 묶으면 **각 fold 안에서 알아서 `fit`/`transform`을 구분**해줍니다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 스케일링이 필요한 모델(로지스틱 회귀)을 교차 검증할 때
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe_scores = cross_val_score(pipe, Xc_train, yc_train, cv=5, scoring="accuracy")

print(f"파이프라인(스케일링+로지스틱): {pipe_scores.mean():.4f}")
print()
print("파이프라인 구조:")
for name, step in pipe.named_steps.items():
    print(f"  {name}: {step}")

---

## 13. `GridSearchCV`와 결과를 읽는 법

1부와 같은 방식으로 분류 모델을 튜닝합니다. 여기서는 **나온 점수를 어떻게 읽어야 하는지**가
더 중요합니다.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 5, 10],
}

gs_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,                  # 5겹 교차 검증
    scoring="accuracy",
    n_jobs=-1,             # 모든 코어 사용
)

gs_dt.fit(Xc_train, yc_train)

print(f"시험한 조합: {len(gs_dt.cv_results_['params'])}개")
print(f"학습 횟수  : {len(gs_dt.cv_results_['params'])} × 5 = "
      f"{len(gs_dt.cv_results_['params']) * 5}회")
print()
print("최적 조합:", gs_dt.best_params_)
print(f"교차 검증 점수: {gs_dt.best_score_:.4f}")

탐색이 끝난 뒤 **찾은 모델을 실제로 어떻게 쓰는지** 봅시다.

In [ ]:
# 학습이 끝나면 최적 모델이 자동으로 전체 학습 데이터에 다시 적합되어 있습니다
best_dt = gs_dt.best_estimator_
print("검증 데이터 정확도:", round(accuracy_score(yc_valid, best_dt.predict(Xc_valid)), 4))
print()

# GridSearchCV 객체 자체를 모델처럼 써도 됩니다 (내부적으로 best_estimator_ 사용)
print("gs_dt.predict()도 동일:", round(accuracy_score(yc_valid, gs_dt.predict(Xc_valid)), 4))

`cv_results_`에는 **시험한 모든 조합의 상세 결과**가 들어 있습니다. 1부 7절에서 본 것과 같은 표이고,
분류라서 점수가 정확도(클수록 좋음)로 나옵니다.

In [ ]:
results = pd.DataFrame(gs_dt.cv_results_)
cols = ["param_max_depth", "param_min_samples_leaf", "param_min_samples_split",
        "mean_test_score", "std_test_score", "rank_test_score"]

results[cols].sort_values("rank_test_score").head(8).round(4)

랜덤 포레스트도 완전히 같은 방식으로 탐색합니다. **모델 클래스와 격자만 바꾸면 됩니다.**

In [ ]:
# 랜덤 포레스트도 같은 방식으로
gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [100, 300],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5, 10],
    },
    cv=5, scoring="accuracy", n_jobs=-1,
)
gs_rf.fit(Xc_train, yc_train)

print("최적 조합:", gs_rf.best_params_)
print(f"교차 검증 점수 : {gs_rf.best_score_:.4f}")
print(f"검증 데이터 점수: {accuracy_score(yc_valid, gs_rf.predict(Xc_valid)):.4f}")

### 결과를 읽을 때 주의할 점

|  | 교차 검증 점수 | 검증 데이터 점수 |
|---|---|---|
| 결정 트리 (튜닝) | 0.8181 | **0.8043** |
| 랜덤 포레스트 (튜닝) | **0.8274** | 0.7826 |

**교차 검증에서는 랜덤 포레스트가 좋았는데, 검증 데이터에서는 결정 트리가 좋습니다.**
모순처럼 보이지만 이유가 있습니다.

- 검증 데이터가 **184건뿐**입니다. 정확도 1%p 차이는 **2명**입니다. 이 정도는 우연히 뒤집힙니다
- `best_score_`는 429건을 5겹으로 나눠 얻은 평균이라 **더 안정적인 추정치**입니다

**데이터가 적을 때는 점수 차이를 과신하면 안 됩니다.** 0.8043 vs 0.7826은
"둘이 비슷하다"고 읽는 것이 맞습니다.

---

## 14. 변수중요도 — 분류에서, 그리고 그 한계

1부에서 변수중요도로 데이터 누출(`fare`)을 잡아냈습니다. 같은 도구를 분류 모델에 적용해보고,
**이 지표를 어디까지 믿어도 되는지**까지 확인합니다.

In [ ]:
fi_clf = plot_importance(rf_clf, Xc_train.columns, top=11,
                         title="생존 예측 — 변수중요도")
fi_clf.round(4)

`age`(0.266)와 `fare`(0.257)가 가장 높고, `who_man`(0.119), `pclass`(0.103)가 뒤따릅니다.

01번에서 본 **[상관계수](../../../glossary.md#correlation) 순위와 다릅니다.** 상관계수는 `adult_male`(-0.557)이 1위였고 `age`는
-0.077로 거의 꼴찌였습니다. 왜 뒤집혔을까요?

- **상관계수는 직선 관계만 봅니다.** 나이와 생존의 관계는 직선이 아닙니다 — 아이는 살아남고,
  청년은 죽고, 노인은 다시 조금 나은 식입니다. 직선으로는 잡히지 않지만
  **트리는 구간을 나눠 이런 관계를 학습합니다**
- 트리는 **다른 변수와의 조합**도 반영합니다. "3등급이면서 성인 남성"처럼요

**두 지표는 서로 다른 것을 재고 있습니다.** 어느 쪽이 맞다기보다, 함께 봐야 그림이 완성됩니다.

### 변수중요도를 믿을 때 주의할 점

`feature_importances_`에는 **알려진 편향**이 있습니다. **고유값이 많은 컬럼을 과대평가**합니다.
값의 종류가 많으면 분할 후보가 많아지고, 그중 하나쯤은 우연히 불순도를 잘 줄이기 때문입니다.

**완전히 무작위인 컬럼을 넣어보면** 이 문제가 드러납니다.

In [ ]:
rng = np.random.default_rng(0)

Xc_noise = Xc_train.copy()
Xc_noise["random_id"] = rng.random(len(Xc_noise))   # 예측과 아무 관계 없는 난수

rf_noise = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_noise, yc_train)

pd.DataFrame({
    "feature": Xc_noise.columns,
    "importance": rf_noise.feature_importances_,
}).sort_values("importance", ascending=False).head(6).round(4)

**아무 의미 없는 난수 컬럼이 3위(0.175)에 올랐습니다.** `pclass`, `sex_male`보다 높습니다.

난수는 학습 데이터 429건의 값이 전부 다르니 분할 지점이 428개나 되고, 그중 몇 개는 우연히 불순도를
줄입니다. 이 "우연한 기여"가 쌓여 중요도로 집계된 것입니다.

### 대안: 순열 중요도 (Permutation Importance)

**한 컬럼의 값만 무작위로 섞은 뒤 성능이 얼마나 떨어지는지** 측정합니다. 성능이 크게
떨어지면 중요한 컬럼이고, 변화가 없으면 안 쓰이던 컬럼입니다.

- **검증 데이터에서 측정**하므로 학습 데이터 암기의 영향을 받지 않습니다
- 어떤 모델에든 쓸 수 있습니다 (트리가 아니어도 됨)
- 여러 번 섞어 평균을 내므로 **느립니다**

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf_clf, Xc_valid, yc_valid, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

compare_imp = pd.DataFrame({
    "feature": Xc_valid.columns,
    "불순도 기반": rf_clf.feature_importances_,
    "순열 중요도": perm.importances_mean,
}).sort_values("순열 중요도", ascending=False)

compare_imp.round(4)

**순위가 크게 달라집니다.**

| 컬럼 | 불순도 기반 | 순열 중요도 |
|---|---|---|
| `age` | 0.266 (1위) | **0.036 (1위)** |
| `who_man` | 0.119 (3위) | **0.031 (2위)** |
| `pclass` | 0.103 (4위) | 0.016 (3위) |
| `fare` | **0.257 (2위)** | **0.003 (6위)** |
| `alone` | 0.013 | **-0.021** |

- **`fare`가 2위에서 6위로 떨어졌습니다.** 불순도 기반에서 높았던 것은 연속값이라
  분할 후보가 많았기 때문이고, 실제로 섞어보니 성능에 별 영향이 없었습니다
- **음수가 나오기도 합니다.** 값을 섞었더니 오히려 성능이 좋아졌다는 뜻으로,
  **그 컬럼이 노이즈에 가깝다**는 신호입니다

**정리하면**

| | 불순도 기반 (`feature_importances_`) | 순열 중요도 |
|---|---|---|
| 속도 | **빠름** (학습하며 계산됨) | 느림 |
| 측정 대상 | 학습 데이터 | **검증 데이터** |
| 편향 | 고유값 많은 컬럼 과대평가 | 상대적으로 공정 |
| 상관된 컬럼 | 중요도가 분산됨 | 둘 다 낮게 나올 수 있음 |

**빠르게 훑을 때는 불순도 기반, 결론을 내릴 때는 순열 중요도**를 쓰면 됩니다.
`fare` 같은 데이터 누출을 찾는 데는 불순도 기반으로도 충분했습니다.

> 마지막 항목도 알아두면 좋습니다. `alone`과 `sibsp`/`parch`처럼 거의 같은 정보를 가진 컬럼이
> 둘 다 있으면, 하나를 섞어도 다른 하나가 대신해서 성능이 안 떨어집니다.
> 그래서 **둘 다 중요하지 않은 것처럼 보일 수 있습니다.** `alone`은 동반 인원이 0명인지를 뜻하니 `sibsp`+`parch`로 그대로 계산됩니다.
> 위 표에서 `alone`이 음수로 나온 이유입니다.

---

## 15. 2부 정리 — 최종 성능

In [ ]:
print("[분류] 생존 예측")
print("  최적 조합:", gs_rf.best_params_)
print(f"  정확도 {accuracy_score(yc_valid, gs_rf.predict(Xc_valid)):.4f}  "
      f"(기준선 {accuracy_score(yc_valid, baseline_clf):.4f})")
print()
print(classification_report(yc_valid, gs_rf.predict(Xc_valid), target_names=["사망", "생존"]))

---

## 정리

**1부 — 택시 (회귀)**

- **결정 트리**는 분산이 가장 많이 줄어드는 질문을 반복해 데이터를 나눕니다.
  잎에 남은 데이터의 **평균**이 예측값입니다
- **깊이 제한이 없으면 학습 데이터를 외웁니다.** 학습 오차 0.001분, 검증 오차는 오히려 악화.
  두 곡선이 벌어지는 지점에서 멈춰야 합니다
- **랜덤 포레스트**는 서로 다른 트리 수백 개의 평균입니다. 부트스트랩 + 컬럼 무작위 선택으로
  다양성을 만듭니다. 튜닝 없이도 잘 동작합니다
- **회귀 지표**: MAE(직관적), RMSE(큰 오차에 민감), R²(평균 대비 얼마나 나은가).
  **반드시 기준선과 비교**해야 의미가 생깁니다
- **변수중요도로 데이터 누출을 찾아냈습니다.** `fare`가 88%를 차지한 것이 단서였고,
  제거하니 MAE 1.19 → 3.55분. 이쪽이 실제로 쓸 수 있는 성능입니다
- **잔차를 구간별로 나눠 보세요.** 전체 MAE가 괜찮아도 특정 구간에서 체계적으로 틀릴 수 있습니다

**2부 — 타이타닉 (분류)**

- 절차는 회귀와 같고 **분할 기준(지니)과 평가 지표만 다릅니다**
- **정확도만 보면 안 됩니다.** 전부 "사망"이라 찍어도 64.7%.
  혼동 행렬 → 정밀도·재현율 → F1 순으로 봅니다. 무엇이 중요한지는 **문제가 정합니다**
- **분류의 교차 검증은 `StratifiedKFold`** 가 자동으로 쓰입니다
- **데이터가 적으면 점수 차이를 과신하지 마세요.** 검증 184건에서 1%p는 2명 차이라 순위가 쉽게 뒤집힙니다. 교차 검증 평균이 더 안정적인 추정치입니다 (`best_score_`를 그대로 최종 성능으로 보고하면 왜 안 되는지는 연습 문제 6번에서 확인합니다)
- **불순도 기반 중요도는 고유값 많은 컬럼을 과대평가합니다.** 난수 컬럼이 3위에 오릅니다.
  결론을 내릴 때는 **순열 중요도**를 함께 보세요

## 스스로 확인해보기

- [ ] 결정 트리가 무엇을 기준으로 데이터를 나누는지 말할 수 있다 (회귀는 분산, 분류는 지니)
- [ ] 학습 오차와 검증 오차가 벌어지는 것이 무슨 뜻인지 안다
- [ ] 회귀 지표를 **기준선과 비교**해야 의미가 생기는 이유를 안다
- [ ] 변수중요도로 데이터 누출을 어떻게 찾아냈는지 설명할 수 있다
- [ ] 정확도만 보면 안 되는 이유를 예로 들 수 있다
- [ ] 정밀도와 재현율 중 무엇이 중요한지는 **문제가 정한다**는 말을 이해했다
- [ ] 데이터가 적을 때 교차 검증 점수와 검증 점수가 엇갈리는 이유를 안다
- [ ] 불순도 기반 중요도의 함정과, 순열 중요도를 함께 봐야 하는 이유를 안다

## 연습 문제

풀어본 뒤 [03_tree_models_solutions.ipynb](03_tree_models_solutions.ipynb)에서 확인하세요.
**문제 1~2는 1부(택시·회귀), 문제 3~6은 2부(타이타닉·분류)** 범위입니다.

### 1부 — 택시 (회귀)

**문제 1.** 이동 시간 예측(누출 제거 버전)에서 `GridSearchCV` 대신
`RandomizedSearchCV`를 써서 더 넓은 범위를 탐색하세요. `n_iter=10`으로 두고,
**소요 시간과 찾아낸 최적 점수**를 `GridSearchCV`와 비교하세요.

**문제 2.** 이동 시간 예측 모델의 잔차를 분석하세요. 어떤 종류의 운행에서 오차가 큰가요?
(힌트: 잔차의 절댓값을 `distance`, `hour`, 실제 `duration` 구간별로 나눠 보세요.)
개선 아이디어를 하나 제안해보세요.

### 2부 — 타이타닉 (분류)

**문제 3.** 타이타닉 분류 모델에서 `max_depth`를 1부터 20까지 바꿔가며 **학습 정확도와 검증
정확도**를 각각 구하고 그래프로 그리세요. 1부의 회귀에서 본 것과 같은 모양이 나오나요?
과적합이 시작되는 깊이는 어디인가요?

**문제 4.** `RandomForestClassifier`에는 `oob_score=True` 옵션이 있습니다.
이것을 켜고 학습한 뒤 `oob_score_`를 확인하고, 5겹 교차 검증 점수와 비교하세요.
**OOB 점수가 무엇인지, 왜 별도의 검증 데이터 없이도 성능을 추정할 수 있는지** 설명해보세요.
(힌트: 부트스트랩 샘플링에서 각 트리는 데이터의 약 63%만 씁니다.)

**문제 5.** 생존 예측에서 **재현율을 최대한 높이고 싶다면** 어떻게 해야 할까요?
아래 두 방법을 각각 적용하고 정밀도·재현율·F1이 어떻게 바뀌는지 비교하세요.
1. `predict_proba`의 임계값을 0.5에서 0.3으로 낮추기
2. `RandomForestClassifier(class_weight="balanced")` 사용하기

**문제 6.** 아래 코드는 에러 없이 실행되지만 **결과를 신뢰할 수 없습니다.** 무엇이 문제일까요?

```python
gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
gs.fit(X_clf, y_clf)                       # 전체 데이터로 탐색
print("최종 성능:", gs.best_score_)         # 이 점수를 최종 성능으로 보고
```

---

다음 노트북: [04_dnn_keras.ipynb](../04_dnn_keras/04_dnn_keras.ipynb) — 같은 데이터를
신경망으로 풀고, 트리 모델과 성능을 비교합니다.